# Cross-dataset raw persona audit

## Excluded persona attributes

Attributes intentionally omitted from `persona_spec.yaml`:

| Attribute | Reason for exclusion |
| --- | --- |
| `census_division` | Excluded as a separate attribute (only natively present in `issp`). ISSP's 9 US Census divisions are kept directly as union levels inside `region` without collapsing. |
| `interview_language` | Excluded (`issp` `INTLANG`, `wvs7` `S_INTLANGUAGE`): survey-administration technical variable (~98% English), not a respondent persona trait. |
| `birth_year` | Excluded (`issp` `BIRTH`, `wvs7` `Q261`): 100% redundant with locked `age`, and pairing `age` with exact `birth_year` leaks survey wave year. |
| `US_DEGR` | Excluded (`issp` 2015-2019): 100% redundant with locked `education` (`EDULEVEL`/`DEGREE`). |
| `X003R` | Excluded (`wvs7`): 6-bin age recode, 100% redundant with locked exact `age` (`Q262`). |
| `EDUCYRS` | Excluded (`issp` years of full-time schooling): covered by locked `education`. |
| `TOPBOT` | Excluded (`issp` 1-10 subjective social standing scale): omitted per user review. |
| `WORK` | Excluded (`issp` paid work history): largely redundant with `employment_status`; omitted per user review. |
| `WRKHRS` | Excluded (`issp` weekly hours worked): omitted per user review. |
| `EMPREL` | Excluded (`issp` employment relationship): omitted per user review. |
| `WRKSUP` | Excluded (`issp` supervises other employees): omitted per user review. |
| `NSUP` | Excluded (`issp` number of employees supervised): omitted per user review. |
| `TYPORG1` | Excluded (`issp` for-profit vs non-profit organization): omitted per user review. |
| `PARTLIV` | Excluded (`issp` living in steady partnership): omitted per user review. |
| `SPWORK` | Excluded (`issp` spouse/partner paid work history): omitted per user review. |
| `SPWRKHRS` | Excluded (`issp` spouse/partner weekly hours worked): omitted per user review. |
| `SPEMPREL` | Excluded (`issp` spouse/partner employment relationship): omitted per user review. |
| `SPWRKSUP` | Excluded (`issp` spouse/partner supervises other employees): omitted per user review. |
| `SPISCO08` | Excluded (`issp` spouse/partner ISCO-08 occupation): omitted per user review. |
| `HHCHILDR` | Excluded (`issp` school-age children in household): omitted per user review. |
| `HHTODD` | Excluded (`issp` toddlers below school entry age in household): omitted per user review. |
| `HHADULT` | Excluded (`issp` adults in household): omitted per user review. |
| `H_SETTLEMENT` | Excluded (`wvs7` settlement type): non-US administrative wording duplicating `town_size` and `urban_rural`; omitted per user review. |
| `VENDING_BEV` | Excluded (`grummon2023` vending machine beverage frequency): narrow study-specific shopping frequency (~70% floor, 86% identical to `VENDING_SNACK`); omitted per user review. |
| `VENDING_SNACK` | Excluded (`grummon2023` vending machine snack frequency): narrow study-specific shopping frequency (~71% floor, 86% identical to `VENDING_BEV`); omitted per user review. |
| `covid__party_lean`, `kreps2021__party_lean` | Excluded (`covid` & `kreps2021` party lean): fielded to all respondents without skip logic, contradicting `party3` for declared partisans (70 cross-party Democrat/Republican rows in `covid`, 43 in `kreps2021`); omitted per user review. |
| `berke2023__Q_v_1` | Excluded (`berke2023` hypothetical prescription delivery willingness): conjoint block screener that gates `CHOICE_v1_*` (0/392 `No` saw block); omitted per user review. |
| `berke2023__Q_v_2` | Excluded (`berke2023` hypothetical liquor store delivery willingness): conjoint block screener that gates `CHOICE_v2_*` (0/1,334 `No` saw block); omitted per user review. |

In [1]:
%load_ext autoreload
%autoreload 2

import importlib
import sys
from pathlib import Path

import pandas as pd

CODE_DIR = Path("../..").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from data import persona_loaders
importlib.reload(persona_loaders)

def show_value_counts(df, col):
    present_ds = [
        ds for ds in dataset_order
        if df.loc[df["dataset"] == ds, col].notna().any()
    ]
    sub = df.loc[df["dataset"].isin(present_ds), ["dataset", col]].copy()
    sub[col] = sub[col].where(sub[col].notna(), "<NA>").astype(str)

    table = pd.crosstab(sub[col], sub["dataset"], margins=True)
    table = table[present_ds + ["All"]]

    has_na = "<NA>" in table.index
    drop_idx = ["All", "<NA>"] if has_na else ["All"]
    body = table.drop(index=drop_idx).sort_values("All", ascending=False)
    na_row = table.loc[["<NA>"]] if has_na else pd.DataFrame(0, index=["<NA>"], columns=table.columns)
    return pd.concat([body, na_row, table.loc[["All"]]])


all_personas = persona_loaders.load_raw_all_personas()
std_personas = persona_loaders.load_all_personas()
dataset_order = list(persona_loaders.RAW_LOADERS.keys())
persona_cols = [c for c in all_personas.columns if c not in {"dataset", "CASEID"}]
ds_counts = {c: int(all_personas.loc[all_personas[c].notna(), "dataset"].nunique()) for c in persona_cols}
shared_cols = [c for c, n in ds_counts.items() if n >= 2]
unique_cols = [c for c, n in ds_counts.items() if n == 1]
unmapped_cols = [c for c in persona_cols if "__" in c]

print(f"Stacked all_personas (raw): {len(all_personas):,} rows x {all_personas.shape[1]} cols")
print(f"Stacked std_personas (canonical): {len(std_personas):,} rows x {std_personas.shape[1]} cols")
print(f"Shared ({len(shared_cols)}): {shared_cols}")
print(f"Unique ({len(unique_cols)}): {unique_cols}")
print(f"Unmapped candidates ({len(unmapped_cols)}): {unmapped_cols}")
all_personas["dataset"].value_counts().reindex(dataset_order)

Stacked all_personas (raw): 72,402 rows x 138 cols
Stacked std_personas (canonical): 72,402 rows x 65 cols
Shared (34): ['country', 'gender', 'age', 'region', 'family_income', 'religion', 'religious_attendance', 'political_party', 'marital_status', 'employment_status', 'household_size', 'urban_rural', 'voted_last_election', 'union_member', 'employer_sector', 'birth_year', 'spouse_employment', 'occupation', 'education', 'interview_language', 'spouse_education', 'political_ideology', 'race', 'citizen', 'state', 'income', 'children_in_household', 'hispanic', 'sexual_orientation', 'residence_type', 'health_insurance', 'evangelical_christian', 'flu_vaccination_frequency', 'nutrition_label_use']
Unique (102): ['EDUCYRS', 'US_DEGR', 'personal_income', 'WORK', 'WRKHRS', 'EMPREL', 'WRKSUP', 'NSUP', 'TYPORG1', 'PARTLIV', 'SPWORK', 'SPWRKHRS', 'SPEMPREL', 'SPWRKSUP', 'SPISCO08', 'HHCHILDR', 'HHTODD', 'TOPBOT', 'ancestry', 'HHADULT', 'town_size', 'H_SETTLEMENT', 'X003R', 'immigrant_status', 'Q264'

dataset
issp               14365
twin2k              2058
wvs7                2596
subpop                22
hints7              7278
shed2025           12934
covid               1971
kreps2021           1096
grummon2023         7945
grummon2025         5636
berke2023           3715
kriner2026          1000
wu2025              1862
kantorowicz2026     2242
magistro2025        6056
forsythe2023        1626
Name: count, dtype: int64

In [2]:
all_personas.columns[:20]

Index(['dataset', 'CASEID', 'country', 'gender', 'age', 'region',
       'family_income', 'religion', 'religious_attendance', 'political_party',
       'marital_status', 'employment_status', 'household_size', 'urban_rural',
       'voted_last_election', 'union_member', 'employer_sector', 'birth_year',
       'spouse_employment', 'occupation'],
      dtype='str')

## 1. Value counts — shared persona columns

In [2]:
show_value_counts(all_personas, "country")

dataset,issp,twin2k,wvs7,subpop,hints7,shed2025,covid,grummon2023,grummon2025,berke2023,kriner2026,wu2025,kantorowicz2026,magistro2025,All
United States,0,2058,2596,22,7278,12934,1971,7945,5636,3715,1000,1862,2242,0,49259
US,14365,0,0,0,0,0,0,0,0,0,0,0,0,3025,17390
Canada,0,0,0,0,0,0,0,0,0,0,0,0,0,3031,3031
<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
All,14365,2058,2596,22,7278,12934,1971,7945,5636,3715,1000,1862,2242,6056,69680


In [4]:
show_value_counts(all_personas, "gender")

dataset,issp,twin2k,wvs7,subpop,covid,grummon2023,grummon2025,berke2023,kriner2026,wu2025,kantorowicz2026,magistro2025,All
gender,,,,,,,,,,,,,
Woman,0,0,0,0,0,4078,3400,0,0,0,0,3277,10755
Female,2197,1044,1206,1,999,0,0,1812,512,973,1127,0,9871
Male,1843,1014,1390,1,949,0,0,1823,483,877,1084,0,9464
Man,0,0,0,0,0,3779,2111,0,0,0,0,2699,8589
2. Female,5709,0,0,0,0,0,0,0,0,0,0,0,5709
1. Male,4586,0,0,0,0,0,0,0,0,0,0,0,4586
Non-binary,0,0,0,0,0,69,112,0,0,0,0,0,181
Other,0,0,0,0,0,0,0,68,0,0,0,79,147
-9. No answer,30,0,0,0,0,0,0,0,0,0,0,0,30


In [5]:
# show_value_counts(all_personas, "age")

In [6]:
show_value_counts(all_personas, "region")

dataset,issp,twin2k,subpop,kriner2026,magistro2025,All
region,,,,,,
5. South Atlantic,2128,0,0,0,0,2128
3. East North Central,1730,0,0,0,0,1730
South,0,0,1,376,1212,1589
9. Pacific,1446,0,0,0,0,1446
2. Middle Atlantic,1144,0,0,0,0,1144
7. West South Central,1108,0,0,0,0,1108
West,0,0,0,235,689,924
"South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)",0,834,0,0,0,834
8. Mountain,817,0,0,0,0,817


In [7]:
# show_value_counts(all_personas, "state")

In [8]:
show_value_counts(all_personas, "education")

dataset,issp,twin2k,wvs7,subpop,covid,grummon2023,grummon2025,kriner2026,wu2025,kantorowicz2026,magistro2025,All
education,,,,,,,,,,,,
Bachelor's degree,0,0,0,0,0,1935,2215,0,0,0,0,4150
Some college or technical school,0,0,0,0,0,1814,1195,0,0,0,0,3009
High school graduate (or GED),0,0,0,0,0,1872,624,0,0,0,0,2496
3. Upper secondary,2391,0,0,0,0,0,0,0,0,0,0,2391
HS or less,0,0,0,0,0,0,0,0,0,0,2374,2374
...,...,...,...,...,...,...,...,...,...,...,...,...
-9. No answer,7,0,0,0,0,0,0,0,0,0,0,7
No answer,2,0,0,0,0,0,0,0,0,0,0,2
9. No answer,1,0,0,0,0,0,0,0,0,0,0,1


In [9]:
# show_value_counts(all_personas, "family_income")

In [10]:
show_value_counts(all_personas, "income")

dataset,subpop,covid,kriner2026,kantorowicz2026,magistro2025,All
income,,,,,,
Middle,0,0,0,0,2996,2996
Higher income,0,0,0,1324,0,1324
High,0,0,0,0,1173,1173
Low,0,0,0,0,1102,1102
Lower income,0,0,0,789,0,789
"$100,000 or more",1,459,202,0,0,662
"$20,000 - $39,999",0,376,161,0,0,537
"$40,000 - $59,999",0,347,169,0,0,516
"Below $20,000",0,328,176,0,0,504


In [11]:
# show_value_counts(all_personas, "race")


In [12]:
show_value_counts(all_personas, "hispanic")

dataset,grummon2023,grummon2025,wu2025,All
hispanic,,,,
No,6647,5049,0,11696
Yes,1297,580,0,1877
None of these,0,0,1629,1629
"Spanish, Hispanic, or Latino",0,0,199,199
<NA>,1,7,34,42
All,7945,5636,1862,15443


In [13]:
# show_value_counts(all_personas, "religion")

In [14]:
# show_value_counts(all_personas, "religious_attendance")

In [15]:
# show_value_counts(all_personas, "political_party")

In [16]:
# show_value_counts(all_personas, "political_ideology")

In [17]:
# show_value_counts(all_personas, "marital_status")

In [18]:
# show_value_counts(all_personas, "employment_status")

In [19]:
# show_value_counts(all_personas, "household_size")

In [20]:
# show_value_counts(all_personas, "children_in_household")

In [21]:
# show_value_counts(all_personas, "children_ever_had")

In [22]:
# show_value_counts(all_personas, "urban_rural")

In [23]:
# show_value_counts(all_personas, "voted_last_election")

In [24]:
# show_value_counts(all_personas, "union_member")

In [25]:
# show_value_counts(all_personas, "residence_type")

In [26]:
# show_value_counts(all_personas, "citizen")

In [27]:
# show_value_counts(all_personas, "interview_language")

In [28]:
# show_value_counts(all_personas, "employer_sector")

In [29]:
# show_value_counts(all_personas, "birth_year") # exclude

In [30]:
# show_value_counts(all_personas, "spouse_education")

In [31]:
# show_value_counts(all_personas, "spouse_employment")

In [32]:
# show_value_counts(all_personas, "nutrition_label_use")

## 2. Value counts — single-dataset persona columns

### 2.1 ISSP (20)

In [33]:
# show_value_counts(all_personas, "EDUCYRS")  # Years of full-time schooling
# exclude

In [34]:
# show_value_counts(all_personas, "ancestry")  # Merged US_ETHN1 + US_ETHN2 ancestry

In [35]:
# show_value_counts(all_personas, "personal_income")  # Personal income (US_RINC)

In [33]:
# show_value_counts(all_personas, "TOPBOT")  # Top-bottom social standing (1-10)

In [37]:
# show_value_counts(all_personas, "WORK")  # Currently, formerly, or never in paid work
# exclude

In [38]:
# show_value_counts(all_personas, "WRKHRS").head(20)  # Hours worked weekly
# exclude

In [39]:
# show_value_counts(all_personas, "EMPREL")  # Employment relationship
# Exclude


In [40]:
# show_value_counts(all_personas, "WRKSUP")  # Supervises other employees
# Exclude

In [41]:
# show_value_counts(all_personas, "NSUP")  # Number of employees supervised
# Exclude


In [42]:
# show_value_counts(all_personas, "TYPORG1")  # For-profit vs non-profit organization
# Exclude


In [43]:
# show_value_counts(all_personas, "occupation")  # Occupation (ISCO-08)

In [44]:
# show_value_counts(all_personas, "PARTLIV")  # Living in steady partnership
# Exclude


In [45]:
# show_value_counts(all_personas, "SPWORK")  # Spouse/partner: paid work history
# Exclude


In [46]:
# show_value_counts(all_personas, "SPWRKHRS")  # Spouse/partner: hours worked weekly
# Exclude


In [47]:
# show_value_counts(all_personas, "SPEMPREL")  # Spouse/partner: employment relationship
# Exclude


In [48]:
# show_value_counts(all_personas, "SPWRKSUP")  # Spouse/partner: supervises other employees
# Exclude


In [49]:
# show_value_counts(all_personas, "SPISCO08")  # Spouse/partner: occupation (ISCO-08)
# Exclude


In [50]:
# show_value_counts(all_personas, "HHCHILDR")  # School-age children in household
# Exclude


In [51]:
# show_value_counts(all_personas, "HHTODD")  # Toddlers below school entry age in household
# Exclude


In [52]:
# show_value_counts(all_personas, "HHADULT")  # Adults in household
# Exclude


In [39]:
show_value_counts(all_personas, "political_ideology")  # ISSP 2023 v46 & 2024 v59 (PARTY_LR dropped)


dataset,issp,twin2k,wvs7,subpop,covid,kriner2026,wu2025,All
political_ideology,,,,,,,,
Moderate,0,582,0,1,660,370,0,1613
5,401,0,615,0,0,0,0,1016
Liberal,0,564,0,1,286,134,0,985
Conservative,0,430,0,1,211,123,0,765
Moderate; middle of the road,0,0,0,0,0,0,590,590
Very liberal,0,345,0,0,0,0,223,568
-8. Can't choose,496,0,0,0,0,0,0,496
Very conservative,0,137,0,0,0,0,323,460
5. 5,446,0,0,0,0,0,0,446


### 2.2 WVS7 (21)

In [3]:
# show_value_counts(all_personas, "town_size")  # Town population size (G_TOWNSIZE)

dataset,wvs7,All
G_TOWNSIZE,,
"20,000-50,000",635,635
"100,000-500,000",569,569
"500,000 and more",385,385
"50,000-100,000",239,239
<NA>,768,768
All,2596,2596


In [ ]:
# show_value_counts(all_personas, "H_SETTLEMENT")  # Settlement type
# exclude

dataset,wvs7,All
"Another city, town (not a regional or district center)",1511,1511
District center,954,954
Village,131,131
<NA>,0,0
All,2596,2596


In [25]:
# show_value_counts(all_personas, "Q263")  # Respondent immigrant status

In [8]:
# show_value_counts(all_personas, "Q264")  # Mother immigrant status
# exclude


In [9]:
# show_value_counts(all_personas, "Q265")  # Father immigrant status
# exclude


In [10]:
# show_value_counts(all_personas, "Q266")  # Country of birth: respondent
# exclude


In [11]:
# show_value_counts(all_personas, "Q267")  # Country of birth: mother
# exclude


In [12]:
# show_value_counts(all_personas, "Q268")  # Country of birth: father
# exclude


In [27]:
# show_value_counts(all_personas, "Q271")  # Lives with parents


In [28]:
# show_value_counts(all_personas, "Q272")  # Language spoken at home

In [14]:
# show_value_counts(all_personas, "Q277")  # Mother highest education
# exclude


In [15]:
# show_value_counts(all_personas, "Q278")  # Father highest education
# exclude


In [29]:
# show_value_counts(all_personas, "Q281")  # Respondent occupational group


In [19]:
# show_value_counts(all_personas, "Q282")  # Spouse occupational group
# exclude


In [20]:
# show_value_counts(all_personas, "Q283")  # Father occupational group (at age 14)
# exclude


In [30]:
# show_value_counts(all_personas, "Q285")  # Chief wage earner in household

In [31]:
# show_value_counts(all_personas, "Q286")  # Family savings during past year


In [ ]:
# show_value_counts(all_personas, "Q287")  # Subjective social class

dataset,wvs7,All
Q287,,
Lower middle class,967,967
Upper middle class,762,762
Working class,615,615
Lower class,183,183
Upper class,34,34
<NA>,35,35
All,2596,2596


In [34]:
# show_value_counts(all_personas, "Q288")  # Scale of household incomes (1-10)
# exclude

In [22]:
# show_value_counts(all_personas, "Q289CS9")  # Detailed religious denomination
# exclude


In [35]:
# show_value_counts(all_personas, "voting_frequency")  # National election voting frequency (Q222)

### 2.3 Grummon2023 (5)

In [5]:
# Prompt: "In general, how healthy is your overall diet? Would you say…"
show_value_counts(all_personas, "diet_quality")  # locked -> diet_quality

dataset,grummon2023,All
Good,3216,3216
Fair,2356,2356
Very good,1463,1463
Excellent,465,465
Poor,445,445
<NA>,0,0
All,7945,7945


In [75]:
# Prompt: "How often do you need to have someone help you when you read instructions, pamphlets, or other written materials from your doctor or pharmacy?"
# show_value_counts(all_personas, "medical_reading_help")  # locked -> medical_reading_help

dataset,grummon2023,All
Never,4201,4201
Rarely,1908,1908
Sometimes,1224,1224
Most of the time,382,382
Always,230,230
<NA>,0,0
All,7945,7945


In [76]:
# Prompt: "In a typical week, how often do you buy beverages from vending machines?"
# show_value_counts(all_personas, "VENDING_BEV")  # dropped (narrow study-specific shopping frequency)
# exclude

dataset,grummon2023,All
Never or less than 1 time per week,5548,5548
1 to 2 times per week,1613,1613
3 to 4 times per week,592,592
5 or more times per week,192,192
<NA>,0,0
All,7945,7945


In [77]:
# Prompt: "In a typical week, how often do you buy snacks from vending machines?"
# show_value_counts(all_personas, "VENDING_SNACK")  # dropped (narrow study-specific shopping frequency)
# exclude

dataset,grummon2023,All
VENDING_SNACK,,
Never or less than 1 time per week,5656,5656
1 to 2 times per week,1470,1470
3 to 4 times per week,615,615
5 or more times per week,181,181
<NA>,23,23
All,7945,7945


In [78]:
# Derived BMI category from height & weight prompts:
#   "How tall are you? Please enter your height in feet and inches."
#   "How much do you weigh? Enter your weight in pounds."
# show_value_counts(all_personas, "bmi")  # locked -> bmi

dataset,grummon2023,All
BMI,,
Healthy weight (18.5 to <25.0),2720,2720
Obese (>=30.0),2514,2514
Overweight (25.0 to <30.0),2385,2385
Underweight (<18.5),312,312
<NA>,14,14
All,7945,7945


### 2.4 Grummon2025 (4)

In [79]:
# Prompt: "Do you or anyone else in your household currently get SNAP (Supplemental Nutrition Assistance Program) or Food Stamps? This includes any SNAP benefits or Food Stamps, even if the amount is small and even if the benefits are received on behalf of children in the household."
# show_value_counts(all_personas, "snap_benefits")  # locked -> snap_benefits

dataset,grummon2025,All
SNAP,,
No,4773,4773
Yes,817,817
Don't know,29,29
<NA>,17,17
All,5636,5636


In [80]:
# Prompt: "Do you or anyone else in your household currently receive WIC benefits?"
# show_value_counts(all_personas, "wic_benefits")  # locked -> wic_benefits

dataset,grummon2025,All
WIC,,
No,5434,5434
Yes,138,138
Don't know,47,47
<NA>,17,17
All,5636,5636


In [ ]:
# Prompt: "The next question is about shopping for groceries online. This includes any groceries you buy through the internet or a mobile app for delivery or pick-up, including from Amazon Fresh, Walmart.com, Instacart, Target.com or your local grocery store. In the past year, about what percentage of your groceries did you buy online?"
# show_value_counts(all_personas, "online_grocery_share")  # locked -> online_grocery_share

dataset,grummon2025,All
ONLINEAMT,,
Less than 25%,3136,3136
25-49%,991,991
75% or more,808,808
50-74%,685,685
<NA>,16,16
All,5636,5636


In [ ]:
# Derived sum (0-4 correct) of 4 Nutrition Facts label literacy questions:
#   1. "If you eat the entire container, how many calories will you eat? Type the number of calories in the box."
#   2. "If you are allowed to eat 60 grams of carbohydrates as a snack, how much ice cream (in cups) could you have? Type the number of cups in the box."
#   3. "Your doctor advises you to reduce the amount of saturated fat in your diet. You usually have 42 g of saturated fat each day, which includes one serving of ice cream. If you stop eating ice cream, how many grams of saturated fat would you be consuming each day? Type the number of grams in the box."
#   4. "If you usually eat 2,500 calories in a day, what percentage of your daily value of calories will you be eating if you eat one serving?"
# show_value_counts(all_personas, "nutrition_literacy_score")  # locked -> nutrition_literacy_score

dataset,grummon2025,All
4 out of 4,3257,3257
3 out of 4,1361,1361
2 out of 4,544,544
1 out of 4,327,327
0 out of 4,147,147
<NA>,0,0
All,5636,5636


### 2.5 Wu2025 (15)

In [6]:
all_personas.groupby('dataset')['political_ideology'].unique().loc['wu2025']

array(['Very liberal', 'Moderate; middle of the road',
       'Very conservative', 'Somewhat liberal', 'Slightly liberal', nan,
       'Slightly conservative', 'Somewhat conservative'], dtype=object)

In [7]:
# Prompt: "When it comes to politics, would you describe yourself as libertarian, socialist, communist, liberal, conservative, centrist, or something else?"
show_value_counts(all_personas, "political_label")  # locked -> political_label

dataset,wu2025,All
political_label,,
Conservative,672,672
Liberal,536,536
Other,299,299
Centrist,125,125
Libertarian,122,122
Socialist,76,76
Communist,26,26
<NA>,6,6
All,1862,1862


In [8]:
# Prompt: "Do you own your primary residence?"
show_value_counts(all_personas, "owns_home")  # locked -> owns_home

dataset,wu2025,All
owns_home,,
Yes,1193,1193
No,667,667
<NA>,2,2
All,1862,1862


In [85]:
# Prompt: "Do you rent your primary residence?"
# show_value_counts(all_personas, "rents_home")  # locked -> rents_home

dataset,wu2025,All
RENTHOME,,
No,1116,1116
Yes,745,745
<NA>,1,1
All,1862,1862


In [86]:
# Prompt: "Which of the following best describes your sexual orientation?"
# show_value_counts(all_personas, "sexual_orientation")  # locked -> sexual_orientation

dataset,wu2025,All
ORIENTATION,,
Heterosexual (straight),1643,1643
Bisexual,110,110
Homosexual (gay),59,59
Prefer not to say,24,24
Other,23,23
<NA>,3,3
All,1862,1862


In [ ]:
# Prompt: "Have you ever served on active duty in the US Armed Forces?"
# show_value_counts(all_personas, "military_service")  # locked -> military_service

dataset,wu2025,All
MILITARY,,
No,1586,1586
Yes,261,261
<NA>,15,15
All,1862,1862


In [88]:
# Prompt: "Does your household use electricity for heat?"
# show_value_counts(all_personas, "electric_heating")  # locked -> electric_heating

dataset,wu2025,All
ELECHEAT,,
Yes,1140,1140
No,668,668
<NA>,54,54
All,1862,1862


In [9]:


all_personas.groupby('dataset')['political_party'].unique().loc['wu2025']


array(['Independent', 'Something else', 'Strong Republican',
       'Strong Democrat', 'Lean Republican', 'Not very strong Democrat',
       'Not very strong Republican', 'Lean Democrat', nan], dtype=object)

In [12]:
# Prompt: "Would you call yourself a strong Republican or a not very strong Republican?" (Republicans only)
# show_value_counts(all_personas, "wu2025__republicanstrength")  # folded -> political_party

In [13]:
# Prompt: "Would you call yourself a strong Democrat or a not very strong Democrat?" (Democrats only)
# show_value_counts(all_personas, "wu2025__democratstrength")  # folded -> political_party

In [15]:
# Prompt: "Do you think of yourself as closer to the Republican or Democratic party?" (Independents/Other only)
# show_value_counts(all_personas, "wu2025__leaner")  # folded -> political_party

In [10]:
# Prompt: "Generally, how interested are you in politics?"
# show_value_counts(all_personas, "political_interest")  # locked -> political_interest

dataset,wu2025,All
wu2025__polint,,
Somewhat interested,557,557
Very interested,491,491
Extremely interested,432,432
Not very interested,232,232
Not at all interested,148,148
<NA>,2,2
All,1862,1862


In [11]:
# Prompt: "Is your electricity bill included directly in your rent?" (renters only)
# show_value_counts(all_personas, "electricity_included_in_rent")  # locked -> electricity_included_in_rent

dataset,wu2025,All
wu2025__electricityinrent,,
No,478,478
Yes,255,255
<NA>,1129,1129
All,1862,1862


In [12]:
# Prompt: "Do you know how much your monthly electricity bill is?"
# show_value_counts(all_personas, "knows_electricity_bill")  # locked -> knows_electricity_bill

dataset,wu2025,All
Yes,1527,1527
No,335,335
<NA>,0,0
All,1862,1862


In [13]:
# Prompt: "Does your household receive energy from an investor-owned utility (IOU)?"
# show_value_counts(all_personas, "uses_investor_owned_utility")  # locked -> uses_investor_owned_utility

dataset,wu2025,All
wu2025__currentiou,,
No,900,900
Yes,413,413
<NA>,549,549
All,1862,1862


In [14]:
# Prompt: "Do you currently participate in a Community Choice Aggregation (CCA)?"
# show_value_counts(all_personas, "uses_community_choice_aggregation")  # locked -> uses_community_choice_aggregation

dataset,wu2025,All
wu2025__currentcca,,
Yes,298,298
No,296,296
<NA>,1268,1268
All,1862,1862


In [15]:
# Prompt: "Do you know what a Community Choice Aggregation (CCA) is?"
# show_value_counts(all_personas, "knows_community_choice_aggregation")  # locked -> knows_community_choice_aggregation

dataset,wu2025,All
wu2025__ccaknowledge,,
No,1231,1231
Yes,615,615
<NA>,16,16
All,1862,1862


### 2.6 HINTS 7 (24)

In [3]:
# show_value_counts(all_personas, "CENSREG")  # Region candidate: 4 US Census regions  # dropped (using CENSDIV)

dataset,hints7,All
South,3325,3325
West,1670,1670
Midwest,1220,1220
Northeast,1063,1063
<NA>,0,0
All,7278,7278


In [ ]:
# show_value_counts(all_personas, "CENSDIV")  # Region candidate: 9 US Census divisions (matches ISSP US_REG)  # locked -> region

In [5]:
# show_value_counts(all_personas, "NCHSURCODE2013")  # Urban/rural candidate: NCHS 2013 6-level metro/fringe/rural  # locked -> urban_rural

dataset,hints7,All
Metropolitan: large metro,2755,2755
Metropolitan: large fringe metro,1590,1590
Metropolitan: medium metro,1439,1439
Non-metropolitan: micropolitan,578,578
Metropolitan: small metro,486,486
Non-metropolitan: noncore,430,430
<NA>,0,0
All,7278,7278


In [6]:
# show_value_counts(all_personas, "PR_RUCA_2010")  # Urban/rural candidate: USDA 2010 RUCA 4-level  # dropped (using NCHSURCODE2013)

dataset,hints7,All
Metropolitan,6245,6245
Micropolitan,582,582
Small town,293,293
Rural,158,158
<NA>,0,0
All,7278,7278


In [7]:
# show_value_counts(all_personas, "RUC2023")  # Urban/rural candidate: USDA 2023 9-level continuum  # dropped (using NCHSURCODE2013)

dataset,hints7,All
Counties in metro areas of 1 million population or more,4420,4420
"Counties in metro areas of 250,000 to 1 million population",1389,1389
"Counties in metro areas of fewer than 250,000 population",468,468
"Nonmetro - Urban population of 20,000 or more, adjacent to a metro area",287,287
"Nonmetro - Urban population of 5,000 to 20,000, adjacent to a metro area",259,259
"Nonmetro - Urban population of fewer than 5,000, adjacent to a metro area",150,150
"Nonmetro - Urban population of 5,000 to 20,000, not adjacent to a metro area",139,139
"Nonmetro - Urban population of 20,000 or more, not adjacent to a metro area",91,91
"Nonmetro - Urban population of fewer than 5,000, not adjacent to a metro area",75,75
<NA>,0,0


In [37]:
# show_value_counts(all_personas, "Education")  # Education candidate: raw R8 (7 levels incl. vocational/technical)  # locked -> education

dataset,hints7,All
Education,,
College graduate,1863,1863
Some college,1445,1445
Postgraduate,1327,1327
12 years or completed high school,1121,1121
Post high school training other than college (vocational or technical),496,496
8 through 11 years,298,298
Less than 8 years,145,145
<NA>,583,583
All,7278,7278


In [38]:
# show_value_counts(all_personas, "EducB")  # Education candidate: derived 5 standard US levels
# exclude?


In [36]:
# show_value_counts(all_personas, "EducA")  # Education candidate: derived 4 coarse levels
# exclude


In [ ]:
# show_value_counts(all_personas, "IncomeRanges")  # Income candidate: raw R16 (9 brackets, observed only)  # locked -> family_income

dataset,hints7,All
IncomeRanges,,
"$100,000 to $199,999",1302,1302
"$50,000 to $74,999",1030,1030
"$20,000 to $34,999",802,802
"$75,000 to $99,999",785,785
"$35,000 to $49,999",782,782
"$200,000 or more",555,555
"$0 to $9,999",494,494
"$10,000 to $14,999",320,320
"$15,000 to $19,999",309,309


In [35]:
# show_value_counts(all_personas, "HHInc")  # Income candidate: derived 6 brackets
# exclude


In [ ]:
# show_value_counts(all_personas, "IncomeRanges_IMP")  # Income candidate: hot-deck imputed 9 brackets (not observed)  # dropped (hot-deck imputed)

dataset,hints7,All
IncomeRanges_IMP,,
"$100,000 to $199,999",1419,1419
"$50,000 to $74,999",1075,1075
"$20,000 to $34,999",908,908
"$35,000 to $49,999",895,895
"$0 to $9,999",847,847
"$75,000 to $99,999",816,816
"$200,000 or more",577,577
"$10,000 to $14,999",382,382
"$15,000 to $19,999",344,344


In [ ]:
# show_value_counts(all_personas, "IncomeFeelings")  # Section R17: subjective feelings about present income  # dropped (subjective financial stress)

dataset,hints7,All
IncomeFeelings,,
Getting by on present income,2485,2485
Living comfortably on present income,2451,2451
Finding it difficult on present income,1115,1115
Finding it very difficult on present income,551,551
<NA>,676,676
All,7278,7278


In [ ]:
# show_value_counts(all_personas, "HINTS_R10_RACE").head(25)  # Race candidate: multi-select from raw R10 checkboxes (separate from hispanic)  # locked -> race

dataset,hints7,All
HINTS_R10_RACE,,
White,4531,4531
Black or African American,1139,1139
Asian,388,388
White; American Indian or Alaska Native,102,102
Native Hawaiian or Other Pacific Islander,69,69
White; Black or African American,59,59
American Indian or Alaska Native,55,55
White; Asian,40,40
White; Black or African American; American Indian or Alaska Native,23,23


In [ ]:
# show_value_counts(all_personas, "Race_Cat2")  # Race candidate: NCI 12-level race recode (keeps Asian origins separate)  # dropped (using HINTS_R10_RACE)

dataset,hints7,All
Race_Cat2,,
White only,4531,4531
Black only,1139,1139
Multiple races selected,311,311
Chinese only,99,99
Asian Indian only,81,81
Other Pacific Islander only,67,67
American Indian or Alaska Native only,55,55
Filipino only,55,55
Other Asian only,45,45


In [ ]:
# show_value_counts(all_personas, "RaceEthn")  # Race candidate: 7-level combined race + Hispanic recode  # dropped (combined with hispanic)

dataset,hints7,All
RaceEthn,,
Non-Hispanic White,3548,3548
Hispanic,1323,1323
Non-Hispanic Black or African American,968,968
Non-Hispanic Asian,342,342
Non-Hispanic Multiple Races Mentioned,218,218
Non-Hispanic Native Hawaiian or other Pacific Islander,21,21
Non-Hispanic American Indian or Alaska Native,18,18
<NA>,840,840
All,7278,7278


In [ ]:
# show_value_counts(all_personas, "RaceEthn5")  # Race candidate: 5-level combined race + Hispanic recode  # dropped (combined with hispanic)

dataset,hints7,All
RaceEthn5,,
Non-Hispanic White,3548,3548
Hispanic,1323,1323
Non-Hispanic Black or African American,968,968
Non-Hispanic Asian,342,342
Non-Hispanic Other,257,257
<NA>,840,840
All,7278,7278


In [33]:
# show_value_counts(all_personas, "HINTS_R5_OCCUPATION").head(25)  # Employment candidate: multi-select from raw R5 checkboxes
# exclude


In [ ]:
# show_value_counts(all_personas, "Occupation2_Cat")  # Employment candidate: single-select recode (687 in Multiple)  # locked -> employment_status

dataset,hints7,All
Occupation2_Cat,,
"Employed, including self-employed only",3108,3108
Retired only,1933,1933
Multiple Occupation statuses selected,687,687
Disabled only,357,357
A homemaker/caregiver only,239,239
Out of work for 1 year or more only,110,110
A student only,101,101
Out of work for less than 1 year only,92,92
Other Occupation only,78,78


In [32]:
# show_value_counts(all_personas, "WorkHrsPerWeek").head(20)  # Section R4: hours worked per week
# exclude


In [30]:
# show_value_counts(all_personas, "SexualOrientation2")  # Section R12: raw sexual orientation
# exclude


In [31]:
# show_value_counts(all_personas, "SexualOrientation_Recode")  # Derived sexual orientation recode  # locked -> sexual_orientation
# use

dataset,hints7,All
SexualOrientation_Recode,,
"Straight, that is, not gay or lesbian",5886,5886
Bisexual,221,221
Lesbian or gay,197,197
Don't know,185,185
Something else,35,35
<NA>,754,754
All,7278,7278


In [29]:
# show_value_counts(all_personas, "EthnicGroupBelonging")  # Section R11: sense of belonging to ethnic/racial group (attitudinal)
# exclude


In [28]:
# show_value_counts(all_personas, "Language_Flag")  # Admin: survey language (English/Spanish/Mixed)
# exclude


In [27]:
# show_value_counts(all_personas, "FormType")  # Admin: survey mode (Paper vs Web)
# exclude

### 2.7 Covid (complete)

In [54]:
# Prompt: "Do you currently have health insurance?"
# show_value_counts(all_personas, "health_insurance")  # locked -> health_insurance

dataset,covid,All
Yes,1672,1672
No,244,244
Prefer not to say,55,55
<NA>,0,0
All,1971,1971


In [ ]:
# Prompt: "Would you describe yourself as a born-again or evangelical Christian, or not?" (asked of Christians)
# Prompt to use: "Born-again or evangelical Christian"
# show_value_counts(all_personas, "evangelical_christian")  # locked -> evangelical_christian

dataset,covid,All
covid__evangelical,,
No,551,551
Yes,435,435
<NA>,985,985
All,1971,1971


In [ ]:
# Prompt: "Have you experienced any recent changes in your employment status?"
# Prompt to use: Recent changes in employment status
# show_value_counts(all_personas, "recent_employment_change")  # locked -> recent_employment_change

dataset,covid,All
covid__work_status,,
No change,801,801
Reduced hours,462,462
Getting laid off,270,270
Prefer not to say,54,54
<NA>,384,384
All,1971,1971


In [ ]:
# Prompt: "Are you able to work from home?"
# show_value_counts(all_personas, "able_to_work_from_home")  # locked -> able_to_work_from_home

dataset,covid,All
covid__work_home,,
Yes,772,772
No,487,487
To some extent,205,205
Prefer not to say,30,30
<NA>,477,477
All,1971,1971


In [58]:
# Prompt: "As of today, do you lean more toward the Democratic Party or the Republican Party?"
# show_value_counts(all_personas, "covid__party_lean")  # excluded (no skip logic; contradicts party3)

dataset,covid,All
Democratic Party,848,848
Republican Party,809,809
Neither/don't know,314,314
<NA>,0,0
All,1971,1971


In [59]:
# Prompt: "How often in the past have you gotten a flu vaccination?"
# show_value_counts(all_personas, "flu_vaccination_frequency")  # locked -> flu_vaccination_frequency

dataset,covid,All
Every year,569,569
Once or twice,569,569
Never,418,418
Most years,415,415
<NA>,0,0
All,1971,1971


In [ ]:
# Prompt: "Do you personally know someone who has been officially tested positive for Covid-19?"
# use original wording
# show_value_counts(all_personas, "knows_someone_tested_positive_covid")  # locked -> knows_someone_tested_positive_covid

dataset,covid,All
No,1131,1131
Yes,840,840
<NA>,0,0
All,1971,1971


In [ ]:
# Prompt: "Do you personally know someone who has been hospitalized or died as a result of having COVID-19?"
# use original wording
# show_value_counts(all_personas, "knows_someone_hospitalized_or_died_covid")  # locked -> knows_someone_hospitalized_or_died_covid

dataset,covid,All
No,1485,1485
Yes,486,486
<NA>,0,0
All,1971,1971


### 2.7b Kreps2021 (complete)

In [14]:
# Prompt: "How often in the past have you gotten a flu vaccination?" (shared with covid)
show_value_counts(all_personas, "flu_vaccination_frequency")  # locked -> flu_vaccination_frequency

dataset,covid,kreps2021,All
Once or twice,569,348,917
Every year,569,335,904
Most years,415,214,629
Never,418,199,617
<NA>,0,0,0
All,1971,1096,3067


In [4]:
# Prompt: "Watching television news programs that report the day's news"
# show_value_counts(all_personas, "tv_news_consumption")  # locked -> tv_news_consumption

dataset,kreps2021,All
A great deal,510,510
A fair amount,335,335
Only a little,168,168
Not at all,83,83
<NA>,0,0
All,1096,1096


In [5]:
# Prompt: "Reading a newspaper (printed or online version)"
# show_value_counts(all_personas, "newspaper_consumption")  # locked -> newspaper_consumption

dataset,kreps2021,All
A great deal,329,329
A fair amount,313,313
Only a little,236,236
Not at all,218,218
<NA>,0,0
All,1096,1096


In [6]:
# Prompt: "Seeing or reading links to news stories on Facebook or other social media sites"
# show_value_counts(all_personas, "social_media_news_consumption")  # locked -> social_media_news_consumption

dataset,kreps2021,All
kreps2021__social_news,,
A great deal,353,353
A fair amount,336,336
Not at all,209,209
Only a little,197,197
<NA>,1,1
All,1096,1096


In [7]:
# Prompt: "I use social media very often every day."
# show_value_counts(all_personas, "social_media_daily_use")  # locked -> social_media_daily_use

dataset,kreps2021,All
Strongly agree,488,488
Agree,367,367
Strongly disagree,96,96
Disagree,91,91
Unsure,54,54
<NA>,0,0
All,1096,1096


In [8]:
# Prompt: "I share photos and videos on social media."
# show_value_counts(all_personas, "social_media_shares_media")  # locked -> social_media_shares_media

dataset,kreps2021,All
Agree,394,394
Strongly agree,315,315
Disagree,158,158
Strongly disagree,158,158
Unsure,71,71
<NA>,0,0
All,1096,1096


In [9]:
# Prompt: "There are social media accounts that I follow constantly."
# show_value_counts(all_personas, "social_media_follows_accounts")  # locked -> social_media_follows_accounts

dataset,kreps2021,All
kreps2021__sns3,,
Agree,410,410
Strongly agree,337,337
Disagree,141,141
Strongly disagree,131,131
Unsure,76,76
<NA>,1,1
All,1096,1096


In [10]:
# Prompt: "I have more than one social media account that I use actively."
# show_value_counts(all_personas, "social_media_multiple_accounts")  # locked -> social_media_multiple_accounts

dataset,kreps2021,All
Agree,330,330
Strongly agree,308,308
Strongly disagree,218,218
Disagree,170,170
Unsure,70,70
<NA>,0,0
All,1096,1096


In [11]:
# Prompt: "I think that social media is very useful."
# show_value_counts(all_personas, "social_media_perceived_usefulness")  # locked -> social_media_perceived_usefulness

dataset,kreps2021,All
kreps2021__sns5,,
Agree,420,420
Strongly agree,307,307
Unsure,198,198
Disagree,97,97
Strongly disagree,72,72
<NA>,2,2
All,1096,1096


In [12]:
# Prompt: "I like to follow my friends' shares on social media."
# show_value_counts(all_personas, "social_media_follows_friends_shares")  # locked -> social_media_follows_friends_shares

dataset,kreps2021,All
kreps2021__sns6,,
Agree,466,466
Strongly agree,345,345
Disagree,102,102
Strongly disagree,93,93
Unsure,89,89
<NA>,1,1
All,1096,1096


In [13]:
# Prompt: "I usually check out what my friends do on social media."
# show_value_counts(all_personas, "social_media_checks_friends_activity")  # locked -> social_media_checks_friends_activity

dataset,kreps2021,All
Agree,444,444
Strongly agree,356,356
Disagree,106,106
Unsure,96,96
Strongly disagree,94,94
<NA>,0,0
All,1096,1096


### 2.8 Berke2023 (complete)

In [62]:
# Prompt: "How often (on average) do you make online shopping purchases?"
# show_value_counts(all_personas, "online_shopping_frequency")  # locked -> online_shopping_frequency

dataset,berke2023,All
Multiple times a month,1893,1893
About once a month,757,757
More than once a week,744,744
Once in a few months or longer,312,312
Never,9,9
<NA>,0,0
All,3715,3715


In [63]:
# Prompt: "How often (on average) do you have take-out food delivered to your home?"
# show_value_counts(all_personas, "takeout_delivery_frequency")  # locked -> takeout_delivery_frequency

dataset,berke2023,All
Once in a few months or longer,1039,1039
Multiple times a month,1019,1019
About once a month,719,719
Never,701,701
More than once a week,237,237
<NA>,0,0
All,3715,3715


In [64]:
# Prompt: "How often (on average) do you purchase your groceries online?"
# show_value_counts(all_personas, "online_grocery_frequency")  # locked -> online_grocery_frequency

dataset,berke2023,All
Never,1138,1138
Once in a few months or longer,922,922
Multiple times a month,884,884
About once a month,619,619
More than once a week,152,152
<NA>,0,0
All,3715,3715


In [65]:
# Prompt: "Suppose there is a company that specializes in delivering prescription medications and offers service in your area, and the total price is similar to local pharmacies. If you needed prescription medications, would you ever order them from this company for delivery to your home?"
# show_value_counts(all_personas, "berke2023__Q_v_1")  # excluded (conjoint block screener gating CHOICE_v1_*)

dataset,berke2023,All
Yes,2737,2737
I don't know,586,586
No,392,392
<NA>,0,0
All,3715,3715


In [66]:
# Prompt: "Would you ever order items from a liquor store for delivery to your home?"
# show_value_counts(all_personas, "berke2023__Q_v_2")  # excluded (conjoint block screener gating CHOICE_v2_*)

dataset,berke2023,All
Yes,1991,1991
No,1334,1334
I don't know,390,390
<NA>,0,0
All,3715,3715


### 2.9 TODO Kriner2026 (7)

In [67]:
# Prompt: "As of today, do you lean more toward the Democratic Party or the Republican Party?" (Independents only)
show_value_counts(all_personas, "kriner2026__party_lean")

dataset,kriner2026,All
kriner2026__party_lean,,
Neither/don't know,150,150
Democratic Party,84,84
Republican Party,82,82
<NA>,684,684
All,1000,1000


In [68]:
# Verasight panel profile (no survey prompt): "2024 presidential general election vote choice"
show_value_counts(all_personas, "kriner2026__vs_votechoice_2024")

dataset,kriner2026,All
kriner2026__vs_votechoice_2024,,
Kamala Harris,300,300
Donald Trump,283,283
Someone else,81,81
<NA>,336,336
All,1000,1000


In [69]:
# Prompt: "Which of the following LEAST describes the place where you live:"
show_value_counts(all_personas, "kriner2026__urban_rural_least")

dataset,kriner2026,All
Rural,498,498
Urban,351,351
Suburban,151,151
<NA>,0,0
All,1000,1000


In [70]:
# Prompt: "Do you approve or disapprove of the way Donald Trump is handling his job as president?"
show_value_counts(all_personas, "kriner2026__approve")

dataset,kriner2026,All
Disapprove,521,521
Approve,359,359
No opinion,120,120
<NA>,0,0
All,1000,1000


In [71]:
# Prompt: "From what you know about global climate change or global warming, which one of the following statements comes closest to your opinion?"
show_value_counts(all_personas, "kriner2026__climate_change_old")

dataset,kriner2026,All
"Global climate change has been established as a serious problem, and immediate action is necessary.",412,412
There is enough evidence that climate change is taking place and some action should be taken.,307,307
"We don't know enough about global climate change, and more research is necessary before we take any actions.",170,170
Concern about global climate change is exaggerated. No action is necessary.,68,68
Global climate change is not occurring; this is not a real issue.,43,43
<NA>,0,0
All,1000,1000


In [72]:
# Prompt: "New technology has significantly increased the production of sustainable aviation fuels made from renewable biomass and waste resources. These sustainable aviation fuels deliver the performance of petroleum-based jet fuel with a fraction of its carbon footprint. This enables airlines to significantly reduce greenhouse gas emissions. How informed are you about sustainable aviation fuels?"
show_value_counts(all_personas, "kriner2026__saf_familiar_old")

dataset,kriner2026,All
Not at all informed,544,544
Somewhat informed,237,237
Moderately informed,143,143
Very informed,76,76
<NA>,0,0
All,1000,1000


In [73]:
# Prompt: "How knowledgeable would you say you are about science generally?"
show_value_counts(all_personas, "kriner2026__sci_know_self")

dataset,kriner2026,All
Somewhat Knowledgeable,386,386
5,169,169
3,106,106
6,106,106
2,99,99
Very Knowledgeable,78,78
Not at All Knowledgeable,56,56
<NA>,0,0
All,1000,1000


### 2.10 TODO Kantorowicz2026 (3)

In [74]:
# No verbatim prompt in paper/SI/archive — undocumented behavioral risk-task classification (Risk averse vs Risk seeking)
show_value_counts(all_personas, "kantorowicz2026__risk_aversion_dummy")

dataset,kantorowicz2026,All
kantorowicz2026__risk_aversion_dummy,,
Risk averse,1362,1362
Risk seeking,381,381
<NA>,499,499
All,2242,2242


In [75]:
# Derived binary CRT score (undocumented cutoff); 3-item CRT prompt:
#   "Please answer the following questions:
#    1. A postcard and a pen cost 150 cents in total. The postcard costs 100 cents more than the pen. How many cents does the pen cost?
#    2. If it takes 3 nurses 3 minutes to measure the blood pressure of 3 patients, how long would it take 300 nurses to measure the blood pressure of 300 patients?
#    3. Sally is making some tea. Every hour, the concentration of the tea doubles. If it takes 8 hours for the tea to be ready, how many hours would it take for the tea to reach half of the final concentration?"
show_value_counts(all_personas, "kantorowicz2026__analytical_thinking_dummy")

dataset,kantorowicz2026,All
kantorowicz2026__analytical_thinking_dummy,,
No,1411,1411
Yes,821,821
<NA>,10,10
All,2242,2242


In [76]:
# No verbatim prompt in paper/SI/archive — undocumented risk-task switching consistency flag (0/1)
show_value_counts(all_personas, "kantorowicz2026__consistent_switch")

dataset,kantorowicz2026,All
0,1255,1255
1,987,987
<NA>,0,0
All,2242,2242


### 2.11 TODO Magistro2025 (3)

In [77]:
# Derived recode (no separate prompt; codebook: "includes both Canadian and US parties. Here leaners are included in the Democratic and Republican parties respectively"; raw `party` already extracted)
show_value_counts(all_personas, "magistro2025__party2")

dataset,magistro2025,All
Other,1820,1820
Democratic,1260,1260
Republican,980,980
CPC,786,786
LPC,738,738
NDP,472,472
<NA>,0,0
All,6056,6056


In [78]:
# Derived recode (no separate prompt; codebook: "education binary, factor" College vs No college; raw `education` already extracted)
show_value_counts(all_personas, "magistro2025__education_bin")

dataset,magistro2025,All
No college,4294,4294
College,1762,1762
<NA>,0,0
All,6056,6056


In [79]:
# Derived recode (no separate prompt; codebook: "binary variable for gender, factor" Woman vs Man, dropping Other; raw `gender` already extracted)
show_value_counts(all_personas, "magistro2025__gender_bin")

dataset,magistro2025,All
magistro2025__gender_bin,,
Woman,3277,3277
Man,2699,2699
<NA>,80,80
All,6056,6056


### 2.12 TODO Forsythe2023 (10)

In [80]:
# Survey PDF 404 on CMU Box; SI footnote p. 5: "We collect information on the gender identity for respondents" (shared candidate -> gender)
show_value_counts(all_personas, "forsythe2023__Clean gender")

dataset,forsythe2023,All
Woman,835,835
Man,779,779
I do not wish to answer,7,7
Other,4,4
Trans,1,1
<NA>,0,0
All,1626,1626


In [81]:
# Survey PDF 404 on CMU Box; SI p. 5: "education level" (shared candidate -> education)
show_value_counts(all_personas, "forsythe2023__Clean education")

dataset,forsythe2023,All
4 year university degree,555,555
Masters degree,310,310
Some university education,227,227
2 year university or trade school degree,226,226
Graduated high school,169,169
Doctoral degree,79,79
4 year university degree (bachelors),49,49
Some secondary education,8,8
I do not wish to answer,3,3
<NA>,0,0


In [82]:
# Survey PDF 404 on CMU Box; SI p. 5: "household income" (shared candidate -> family_income)
show_value_counts(all_personas, "forsythe2023__Clean Income Entry")

dataset,forsythe2023,All
"$50,000 to $62,499",204,204
"$62,500 to $74,999",175,175
"$37,500 to $49,999",173,173
"$87,500 to $99,999",154,154
"$75,000 to $87,499",148,148
"$100,000 to $119,999",116,116
"$30,000 to $37,499",107,107
"$140,000 to $159,999",92,92
"$120,000 to $139,999",85,85
"$25,000 to $29,999",72,72


In [83]:
# Survey PDF 404 on CMU Box; SI p. 5: "marital status" (shared candidate -> marital_status)
show_value_counts(all_personas, "forsythe2023__Clean livingSit")

dataset,forsythe2023,All
Married or living with a partner,1048,1048
Single,318,318
Divorced,159,159
Widowed,81,81
Separated,14,14
I do not wish to answer,6,6
<NA>,0,0
All,1626,1626


In [84]:
# Survey PDF 404 on CMU Box; SI Table S3: "Household Size" (shared candidate -> household_size)
show_value_counts(all_personas, "forsythe2023__Clean householdSize")

dataset,forsythe2023,All
forsythe2023__Clean householdSize,,
2,668,668
1,327,327
3,268,268
4,229,229
5,89,89
6,27,27
7,11,11
7 or more,3,3
I do not wish to answer,2,2


In [85]:
# Screener / choice segment (SI p. 4: recent/intended buyers of passenger cars vs SUVs)
show_value_counts(all_personas, "forsythe2023__Clean Vehicle Type")

dataset,forsythe2023,All
SUV,877,877
Car,749,749
<NA>,0,0
All,1626,1626


In [86]:
# Survey PDF 404 on CMU Box; column `Clean elecVehTypeOwned`: vehicle powertrain types owned (multi-select)
show_value_counts(all_personas, "forsythe2023__Clean elecVehTypeOwned")

dataset,forsythe2023,All
forsythe2023__Clean elecVehTypeOwned,,
Conventional,1354,1354
Conventional;Hybrid,162,162
Hybrid,34,34
I have never owned a vehicle,17,17
Conventional;Electric,10,10
Conventional;Hybrid;Plug-in Hybrid,9,9
Electric,9,9
Conventional;Plug-in Hybrid,8,8
Conventional;Hybrid;Electric,5,5


In [87]:
# SI p. 25: "We ask individuals in the survey in which of the following ways they were impacted by the COVID-19 pandemic: Permanent job loss, Temporary job loss, Salary reduction, None of the above."
show_value_counts(all_personas, "forsythe2023__Clean covid")

dataset,forsythe2023,All
forsythe2023__Clean covid,,
None of the above,1186,1186
Salary reduction,187,187
Temporary job loss,117,117
Permanent job loss,69,69
Temporary job loss;Salary reduction,42,42
Permanent job loss;Salary reduction,12,12
Permanent job loss;Temporary job loss;Salary reduction,9,9
Permanent job loss;Temporary job loss,3,3
<NA>,1,1


In [88]:
# Survey PDF 404 on CMU Box; SI p. 5: respondent age (exact integer years; shared candidate -> age)
all_personas.loc[all_personas["dataset"] == "forsythe2023", "forsythe2023__Clean Age"].describe()

count    1623.000000
mean       52.158965
std        17.333513
min        18.000000
25%        36.000000
50%        51.000000
75%        68.000000
max        89.000000
Name: forsythe2023__Clean Age, dtype: float64

In [89]:
# Survey PDF 404 on CMU Box; respondent US 5-digit ZIP code (ZIP.fillna(Clean ZIP))
z = all_personas.loc[all_personas["dataset"] == "forsythe2023", "forsythe2023__ZIP_combined"]
print(f"non-null={int(z.notna().sum())}/{len(z)}, unique={int(z.nunique())}")

non-null=1626/1626, unique=1350
